# Modelo lineal con TF-IDF y Logistic Regression

Este es mi baseline. La idea es montar algo sencillo y rápido para tener una referencia con la que comparar los modelos pesados, y de paso validar que mi esquema de folds y de submission funciona antes de gastar horas de GPU en un transformer.

In [ ]:
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from scipy.sparse import hstack, csr_matrix

TRAIN_PATH = '/kaggle/input/2025-26-false-political-claim-detection/train.csv'
TEST_PATH  = '/kaggle/input/2025-26-false-political-claim-detection/test_nolabel.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
y = train['label'].values

Cargo los CSV directamente desde el dataset de Kaggle, igual que hice durante toda la competición.

In [ ]:
def build_text(row):
    parts = []
    for col, prefix in [('speaker','Speaker'), ('party_affiliation','Party'),
                        ('speaker_job','Job'), ('state_info','State'),
                        ('subject','Topic')]:
        v = row.get(col)
        if pd.notna(v) and str(v).strip() != '':
            parts.append(f'{prefix}: {v}')
    head = ' | '.join(parts)
    return f"{head} [SEP] Claim: {row['statement']}"

train['text_enriched'] = train.apply(build_text, axis=1)
test['text_enriched']  = test.apply(build_text, axis=1)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = np.zeros(len(train), dtype=int)
for f, (_, vi) in enumerate(skf.split(train, y)):
    folds[vi] = f

Replico exactamente la misma función build_text y la misma asignación de folds (semilla 42) del notebook de preprocessing, para que las predicciones OOF de este modelo sean combinables con las de los siguientes.

In [ ]:
vec_w = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.95, sublinear_tf=True, max_features=80000)
vec_c = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), min_df=2, max_df=0.95, sublinear_tf=True, max_features=80000)

Xw_tr = vec_w.fit_transform(train['text_enriched'])
Xw_te = vec_w.transform(test['text_enriched'])
Xc_tr = vec_c.fit_transform(train['text_enriched'])
Xc_te = vec_c.transform(test['text_enriched'])
print(Xw_tr.shape, Xc_tr.shape)

Combino dos vectorizadores porque cada uno captura cosas distintas: palabra 1-2 captura unigramas y bigramas semánticos, y carácter 3-5 me da robustez frente a variantes ortográficas y nombres propios (frecuentes en política). sublinear_tf aplica 1+log(tf), que estabiliza la regresión logística cuando hay términos muy repetidos.

In [ ]:
def smoothed_te(train_df, test_df, col, y, folds, alpha=20.0):
    oof = np.zeros(len(train_df))
    glob = y.mean()
    for f in range(folds.max()+1):
        tr = folds != f
        va = folds == f
        means = pd.Series(y[tr]).groupby(train_df[col].values[tr]).agg(['mean','count'])
        smooth = (means['mean']*means['count'] + glob*alpha) / (means['count']+alpha)
        oof[va] = pd.Series(train_df[col].values[va]).map(smooth).fillna(glob).values
    means_full = pd.Series(y).groupby(train_df[col].values).agg(['mean','count'])
    smooth_full = (means_full['mean']*means_full['count'] + glob*alpha) / (means_full['count']+alpha)
    te = test_df[col].map(smooth_full).fillna(glob).values
    return oof, te

cat_cols = ['speaker','party_affiliation','speaker_job','state_info','subject']
te_tr = np.zeros((len(train), len(cat_cols)))
te_te = np.zeros((len(test),  len(cat_cols)))
for i, c in enumerate(cat_cols):
    te_tr[:,i], te_te[:,i] = smoothed_te(train, test, c, y, folds)
print('target encoding listo')

Para las cinco categóricas elijo target encoding suavizado en lugar de one-hot porque speaker tiene miles de valores distintos y el one-hot acabaría siendo dispersísimo. El suavizado bayesiano (alpha=20) mezcla la media local con la global para que las categorías con pocas observaciones no metan ruido, y el cálculo out-of-fold evita que la fila se vea a sí misma.

In [ ]:
len_tr = csr_matrix(np.array([len(t.split()) for t in train['text_enriched']]).reshape(-1,1))
len_te = csr_matrix(np.array([len(t.split()) for t in test['text_enriched']]).reshape(-1,1))

X_tr = hstack([Xw_tr, Xc_tr, csr_matrix(te_tr), len_tr]).tocsr()
X_te = hstack([Xw_te, Xc_te, csr_matrix(te_te), len_te]).tocsr()
print(X_tr.shape)

In [ ]:
oof = np.zeros(len(train))
test_pred = np.zeros(len(test))

for f in range(5):
    tr = folds != f
    va = folds == f
    clf = LogisticRegression(C=1.0, max_iter=2000, class_weight='balanced', solver='liblinear')
    clf.fit(X_tr[tr], y[tr])
    oof[va] = clf.predict_proba(X_tr[va])[:,1]
    test_pred += clf.predict_proba(X_te)[:,1] / 5
    print(f'fold {f} ok')

Uso liblinear porque va bien con matrices dispersas y problemas binarios. class_weight='balanced' lo elegí en lugar de re-muestrear porque preserva la distribución original y simplemente reescala la pérdida, así las probabilidades del modelo siguen siendo interpretables y reutilizables en ensembles.

In [ ]:
best_t, best_f1 = 0.5, 0
for t in np.arange(0.30, 0.70, 0.01):
    s = f1_score(y, (oof>t).astype(int), average='macro')
    if s > best_f1: best_f1, best_t = s, t
print(f'mejor F1 OOF = {best_f1:.4f}  en threshold {best_t:.2f}')

El threshold por defecto es 0,5 pero con desbalance casi nunca es óptimo. Lo busco en una rejilla sobre las predicciones OOF, sobre el test no podría hacerlo porque no tengo labels.

In [ ]:
np.save('oof_lr.npy', oof)
np.save('test_lr.npy', test_pred)

sub = pd.read_csv('/kaggle/input/2025-26-false-political-claim-detection/sample_submission.csv')
sub['label'] = (test_pred > best_t).astype(int)
sub.to_csv('submission_lr.csv', index=False)
sub.head()

Guardo las probabilidades OOF y test en .npy para poder usarlas como entrada de un stacking o voting más adelante. La submission usa el umbral óptimo que acabo de encontrar.